# MOFSemNet — server experiment notebook (single file)

Self-contained: it does **not** import anything from the MOFSemNet repository. Everything
(data loading, features, graph building, Black Hole / random / k-center sparsification, the GNNs,
the baselines, result export) is in this notebook, so the whole thing can be deleted later.

**What to upload next to this notebook**
1. `MOFCSD.csv` (from `data/raw/` in the repo)
2. `chemberta_embeddings.npz` (from `data/embeddings/`) — optional but recommended. If missing,
   the notebook computes the embeddings itself (needs internet access to Hugging Face).
3. `molt5_embeddings.npz` — only if you want MolT5.

**How to run:** edit the *Configuration* cell (pick a `PROFILE`), then *Run All*.
It prints how many runs it will do first. Results are appended to a CSV after **every single run**,
so if the server kills the job you just run the notebook again and it **resumes** where it stopped.

**Outputs** (folder `mofsemnet_results/`, also zipped for download):
`per_run_results.csv` (one row per run + seed), `aggregated_results.csv` (mean ± std over seeds),
`graph_vs_no_graph.csv`, `mofsemnet_results.xlsx` (all sheets), `run_metadata.json`, `README.txt`.

**Reproducibility:** every row stores the seed, the test-set name, the exact configuration and the
fingerprint (SHA-256) of the data file. Same notebook + same data + same seeds = same numbers
(GPU runs can differ in the last decimals because some GPU operations are not bit-exact).

## 1. Install / imports

In [2]:
# Uncomment on a fresh server (PyTorch Geometric needs a torch build that matches your CUDA):
!pip install -q pandas numpy networkx scikit-learn rdkit tqdm openpyxl transformers
!pip install -q torch_geometric

import hashlib, itertools, json, os, platform, random, sys, time, warnings, zipfile
from datetime import datetime

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from networkx.algorithms.community import louvain_communities
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch_geometric.nn import GATConv, GCNConv
from tqdm.auto import tqdm

RDLogger.DisableLog("rdApp.*")
warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch", torch.__version__)


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python3 -m pip install --upgrade pip
device: cuda | torch 2.9.1+cu128


## 2. Configuration  ← the only cell you normally need to edit

Pick a `PROFILE`, then optionally override single settings in the `CFG[...] = ...` lines below it.

| profile | meaning | rough size |
|---|---|---|
| `smoke` | 800 MOFs, 2 seeds, tiny grid — checks the notebook works | a few minutes |
| `quick` | 3 seeds, k ∈ {3,10}, 3 sparsification methods | hours on CPU, much less on GPU |
| `standard` | 5 seeds, k ∈ {3,5,10}, all methods, retention 30/50 % | the main run |
| `full` | 10 seeds, k ∈ {3,5,10}, retention 30/50/70 %, 3 Black Hole weightings, hybrid β ∈ {0.25,0.5,0.75}, scaling on/off, 2 test sets, GAT | very large — trim it first |

In [3]:
PROFILE = "standard"                       # "smoke" | "quick" | "standard" | "full"

DATA_PATH = "MOFCSD.csv"
EMB_PATHS = {"chemberta": "chemberta_embeddings.npz", "molt5": "molt5_embeddings.npz"}
HF_IDS = {"chemberta": "seyonec/ChemBERTa-zinc-base-v1", "molt5": "laituan245/molt5-base"}
OUT_DIR = "mofsemnet_results"

PROFILES = {
    "smoke": dict(
        n_subsample=800, seeds=[0, 1], language_models=["chemberta"], test_sets=["random15"], scalings=["none"],
        feature_sets=["fingerprint", "fingerprint+embedding"], node_features=["fingerprint"],
        non_graph_models=["random_forest", "knn"], gnn_models=["gcn", "residual_gcn"],
        graph_types=["cs2", "semantic"], k_values=[3], hybrid_betas=[0.5],
        sparsify_graphs=["cs2", "semantic"], sparsify_k_values=[3],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.5],
        bh_weights=[(0.33, 0.33, 0.33)], epochs=30),
    "quick": dict(
        n_subsample=None, seeds=[0, 1, 2], language_models=["chemberta"], test_sets=["random15"], scalings=["none"],
        feature_sets=["fingerprint", "embedding", "fingerprint+embedding"], node_features=["fingerprint"],
        non_graph_models=["random_forest", "mlp"], gnn_models=["gcn", "graphsage", "residual_gcn"],
        graph_types=["cs2", "fingerprint", "semantic", "hybrid"], k_values=[3, 10], hybrid_betas=[0.5],
        sparsify_graphs=["cs2", "fingerprint", "semantic", "hybrid"], sparsify_k_values=[3],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.3, 0.5],
        bh_weights=[(0.33, 0.33, 0.33)], epochs=200),
    "standard": dict(
        n_subsample=None, seeds=[0, 1, 2, 3, 4], language_models=["chemberta"], test_sets=["random15"], scalings=["none"],
        feature_sets=["fingerprint", "embedding", "fingerprint+embedding"], node_features=["fingerprint"],
        non_graph_models=["random_forest", "mlp", "knn"], gnn_models=["gcn", "graphsage", "residual_gcn"],
        graph_types=["cs2", "fingerprint", "semantic", "hybrid"], k_values=[3, 5, 10], hybrid_betas=[0.5],
        sparsify_graphs=["cs2", "fingerprint", "semantic", "hybrid"], sparsify_k_values=[3, 10],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.3, 0.5],
        bh_weights=[(0.33, 0.33, 0.33)], epochs=200),
    "full": dict(
        n_subsample=None, seeds=list(range(10)), language_models=["chemberta", "molt5"],
        test_sets=["random15", "cs2_hubs"], scalings=["none", "standard"],
        feature_sets=["fingerprint", "embedding", "fingerprint+embedding"],
        node_features=["fingerprint", "fingerprint+embedding"],
        non_graph_models=["random_forest", "mlp", "knn"], gnn_models=["gcn", "graphsage", "gat", "residual_gcn"],
        graph_types=["cs2", "fingerprint", "semantic", "hybrid"], k_values=[3, 5, 10], hybrid_betas=[0.25, 0.5, 0.75],
        sparsify_graphs=["cs2", "fingerprint", "semantic", "hybrid"], sparsify_k_values=[3, 5, 10],
        sparsify_methods=["black_hole", "random", "k_center"], retentions=[0.3, 0.5, 0.7],
        bh_weights=[(0.33, 0.33, 0.33), (0.6, 0.2, 0.2), (0.2, 0.6, 0.2)], epochs=200),
}
CFG = dict(PROFILES[PROFILE])

# ---- Settings shared by every profile (edit freely) -------------------------------------------
CFG.update(
    # which experiment blocks to run
    block_non_graph=True,     # RF / MLP / kNN on fingerprint, embedding, fingerprint+embedding  (no graph)
    block_no_edge=True,       # the GNNs with NO edges  -> the "does the graph help or hurt?" reference
    block_graph_full=True,    # GNNs on the full (unpruned) graphs, every graph type x k
    block_sparsify=True,      # Black Hole / random / k-center pruning on the chosen graphs
    # GNN training settings (same as the repo's Step 5)
    hidden=64, lr=0.005, weight_decay=1e-3, patience=20,
    # baselines
    rf_trees=100, knn_neighbors=5, mlp_hidden=(128, 64),
    # splits: 15 % test (fixed, split_state=42), 15 % of the rest for early stopping
    split_state=42, test_frac=0.15, val_frac=0.15,
    # safety valve: stop politely after this many hours (re-run the notebook to resume). None = no limit
    max_hours=None,
)
# Examples of quick overrides (uncomment):
# CFG["seeds"] = list(range(10))
# CFG["k_values"] = [2, 3, 5]
# CFG["retentions"] = [0.2, 0.3, 0.5, 0.7]
# CFG["gnn_models"] = ["gcn", "residual_gcn"]
# CFG["scalings"] = ["none", "standard"]        # standardising features is a cheap, often large win for GNNs
# CFG["language_models"] = ["chemberta", "molt5"]

print(f"PROFILE={PROFILE}")
for key, val in CFG.items():
    print(f"  {key}: {val}")

PROFILE=standard
  n_subsample: None
  seeds: [0, 1, 2, 3, 4]
  language_models: ['chemberta']
  test_sets: ['random15']
  scalings: ['none']
  feature_sets: ['fingerprint', 'embedding', 'fingerprint+embedding']
  node_features: ['fingerprint']
  non_graph_models: ['random_forest', 'mlp', 'knn']
  gnn_models: ['gcn', 'graphsage', 'residual_gcn']
  graph_types: ['cs2', 'fingerprint', 'semantic', 'hybrid']
  k_values: [3, 5, 10]
  hybrid_betas: [0.5]
  sparsify_graphs: ['cs2', 'fingerprint', 'semantic', 'hybrid']
  sparsify_k_values: [3, 10]
  sparsify_methods: ['black_hole', 'random', 'k_center']
  retentions: [0.3, 0.5]
  bh_weights: [(0.33, 0.33, 0.33)]
  epochs: 200
  block_non_graph: True
  block_no_edge: True
  block_graph_full: True
  block_sparsify: True
  hidden: 64
  lr: 0.005
  weight_decay: 0.001
  patience: 20
  rf_trees: 100
  knn_neighbors: 5
  mlp_hidden: (128, 64)
  split_state: 42
  test_frac: 0.15
  val_frac: 0.15
  max_hours: None


## 3. Data and features

In [4]:
INVALID_SMILES, INVALID_REPLACEMENT = "F[Si](F)(F)(F)(F)F", "c1ccccc1"


def derive_pld_category(pld):
    if pd.isna(pld) or pld < 2.4:
        return "nonporous"
    if pld < 4.0:
        return "small pore"
    if pld < 8.0:
        return "medium pore"
    return "large pore"


def load_data(path, n_subsample=None):
    df = pd.read_csv(path, index_col=0)
    df["linker SMILES"] = df["linker SMILES"].replace(INVALID_SMILES, INVALID_REPLACEMENT)
    df = df.rename(columns={"linker SMILES": "linker_smiles", "Pore Limiting Diameter": "pld_value"})
    df["refcode"] = df.index
    df["pld_category"] = df["pld_value"].apply(derive_pld_category)
    if n_subsample:
        df = df.sample(n=min(n_subsample, len(df)), random_state=0)
    return df.reset_index(drop=True)


def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)


def morgan_matrix(smiles_series):
    """1024-bit Morgan fingerprints (radius 2); invalid SMILES -> all-zero row. Computed once per
    distinct SMILES (only ~3,200 distinct linkers exist for 14,296 MOFs)."""
    cache = {}
    for smi in smiles_series.unique():
        mol = Chem.MolFromSmiles(smi)
        cache[smi] = (np.zeros(1024, np.float32) if mol is None or mol.GetNumAtoms() == 0
                      else _MORGAN.GetFingerprintAsNumPy(mol).astype(np.float32))
    return np.stack([cache[s] for s in smiles_series.values]).astype(np.float32)


def build_fingerprint_features(df, fp):
    """1024 Morgan bits + 2 pore-geometry columns + one-hot metal. PLD itself is NOT a feature."""
    pore = df[["Largest Cavity Diameter", "Largest Free Sphere"]].values.astype(np.float32)
    metals = sorted(df["metal"].dropna().unique().tolist())
    index = {m: i for i, m in enumerate(metals)}
    one_hot = np.zeros((len(df), len(metals)), np.float32)
    for row, m in enumerate(df["metal"].values):
        one_hot[row, index[m]] = 1.0
    return np.concatenate([fp, pore, one_hot], axis=1).astype(np.float32)


def embed_with_hf(smiles_list, lm, batch_size=64):
    from transformers import AutoModel, AutoTokenizer, T5EncoderModel
    tok = AutoTokenizer.from_pretrained(HF_IDS[lm])
    model = (T5EncoderModel if lm == "molt5" else AutoModel).from_pretrained(HF_IDS[lm]).to(DEVICE).eval()
    chunks = []
    with torch.no_grad():
        for i in tqdm(range(0, len(smiles_list), batch_size), desc=f"embedding ({lm})"):
            enc = tok(smiles_list[i:i + batch_size], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
            hidden = model(**enc).last_hidden_state
            mask = enc["attention_mask"].unsqueeze(-1).float()
            chunks.append(((hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)).cpu().numpy())
    return np.concatenate(chunks).astype(np.float32)


def get_embeddings(df, lm):
    """Looks every SMILES up in the uploaded .npz (keyed by SMILES text, so row order and
    subsampling do not matter); only SMILES missing from the file are computed with the model."""
    lookup = {}
    path = EMB_PATHS[lm]
    if os.path.exists(path):
        cached = np.load(path, allow_pickle=True)
        for smi, vec in zip(cached["smiles"], cached["embeddings"]):
            lookup.setdefault(str(smi), vec)
        print(f"[{lm}] loaded {len(lookup)} cached embeddings from {path}")
    missing = [s for s in df["linker_smiles"].unique() if s not in lookup]
    if missing:
        print(f"[{lm}] computing {len(missing)} missing embeddings with {HF_IDS[lm]} (needs internet)")
        for smi, vec in zip(missing, embed_with_hf(missing, lm)):
            lookup[smi] = vec
    return np.stack([lookup[s] for s in df["linker_smiles"].values]).astype(np.float32)


df = load_data(DATA_PATH, CFG["n_subsample"])
N = len(df)
LABELS = df["pld_value"].values.astype(np.float32)
DATA_SHA = file_sha256(DATA_PATH)
FP = morgan_matrix(df["linker_smiles"])
X_FP = build_fingerprint_features(df, FP)
EMB = {lm: get_embeddings(df, lm) for lm in CFG["language_models"]}
print(f"MOFs={N}  fingerprint-features={X_FP.shape}  data sha256={DATA_SHA[:16]}…  PLD std={LABELS.std():.3f}")

[chemberta] computing 3193 missing embeddings with seyonec/ChemBERTa-zinc-base-v1 (needs internet)


tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

embedding (chemberta):   0%|          | 0/50 [00:00<?, ?it/s]

MOFs=14296  fingerprint-features=(14296, 1079)  data sha256=e23a42e742099b4c…  PLD std=2.931


## 4. Graph construction  (cs2 = Case Study 2 graph · fingerprint · semantic · hybrid)

* `cs2`: 0.9 × Tanimoto(linker) + 0.1 × same-metal indicator, k nearest neighbours (Case Study 2 / MOFGalaxyNet style)
* `fingerprint`: Tanimoto(linker fingerprints) only
* `semantic`: cosine similarity of the language-model embeddings
* `hybrid`: β × cosine(embeddings) + (1 − β) × Tanimoto

In [5]:
BLOCK = 1500


def _normalize(a):
    norms = np.linalg.norm(a, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return a / norms


def _tanimoto_block(rows):
    inter = FP[rows] @ FP.T
    union = FP[rows].sum(1)[:, None] + FP.sum(1)[None, :] - inter
    union[union == 0] = 1.0
    return inter / union


def knn_graph(sim_fn, k):
    """Undirected top-k graph from a blocked similarity function (never builds the N x N matrix)."""
    g = nx.Graph()
    g.add_nodes_from(range(N))
    kk = min(k, N - 1)
    for start in range(0, N, BLOCK):
        rows = slice(start, min(start + BLOCK, N))
        sim = sim_fn(rows)
        local = np.arange(sim.shape[0])
        sim[local, local + start] = -1.0
        top = np.argpartition(-sim, kk - 1, axis=1)[:, :kk]
        for r in range(sim.shape[0]):
            for c in top[r]:
                if sim[r, c] > -1.0:
                    g.add_edge(start + r, int(c), weight=float(sim[r, c]))
    return g


_METAL_CODES = pd.factorize(df["metal"])[0]
_GRAPH_CACHE = {}


def get_graph(graph_type, k, beta=0.5, lm="-"):
    if graph_type == "none":
        g = nx.Graph()
        g.add_nodes_from(range(N))
        return g
    key = (graph_type, k, beta if graph_type == "hybrid" else 0.0, lm if graph_type in ("semantic", "hybrid") else "-")
    if key not in _GRAPH_CACHE:
        t = time.time()
        if graph_type == "cs2":
            fn = lambda r: 0.9 * _tanimoto_block(r) + 0.1 * (_METAL_CODES[r][:, None] == _METAL_CODES[None, :]).astype(np.float32)
        elif graph_type == "fingerprint":
            fn = _tanimoto_block
        elif graph_type == "semantic":
            en = _normalize(EMB[lm])
            fn = lambda r: en[r] @ en.T
        elif graph_type == "hybrid":
            en = _normalize(EMB[lm])
            fn = lambda r: beta * (en[r] @ en.T) + (1 - beta) * _tanimoto_block(r)
        else:
            raise ValueError(graph_type)
        _GRAPH_CACHE[key] = knn_graph(fn, k)
        print(f"built graph {key}: {_GRAPH_CACHE[key].number_of_edges()} edges in {time.time() - t:.0f}s")
    return _GRAPH_CACHE[key]

## 5. Test sets and splits

* `random15` — 15 % of the MOFs chosen at random with a fixed seed (the Step 5 test set)
* `cs2_hubs` — Case Study 2's test set: ~4 % of the highest-"gravity" MOFs in each Louvain community of the `cs2` k=3 graph

In [6]:
def gravity_per_community(graph, communities, weights):
    wd, wb, ws = weights
    gravity, fixed_test = {}, set()
    for community in communities:
        sub = graph.subgraph(community)
        if sub.number_of_nodes() == 0:
            continue
        deg = nx.degree_centrality(sub)
        bet = nx.betweenness_centrality(sub, normalized=True)
        wsum = {n: sum(d["weight"] for _, _, d in sub.edges(n, data=True)) for n in sub.nodes()}
        nodes = list(community)
        norm = lambda vals: MinMaxScaler().fit_transform(np.array(vals).reshape(-1, 1)).flatten()
        nd, nb, nw = norm([deg[n] for n in nodes]), norm([bet[n] for n in nodes]), norm([wsum[n] for n in nodes])
        for i, n in enumerate(nodes):
            gravity[n] = wd * nd[i] + wb * nb[i] + ws * nw[i]
        degrees = {n: sub.degree(n) for n in community}
        eligible = [n for n, d in degrees.items() if d > 2] or ([max(degrees, key=degrees.get)] if degrees else [])
        if eligible:
            ranked = sorted(eligible, key=lambda n: gravity[n], reverse=True)
            fixed_test.update(ranked[:max(1, int(0.04 * len(community)))])
    return gravity, fixed_test


_TEST_CACHE = {}


def get_test_nodes(name):
    if name not in _TEST_CACHE:
        if name == "random15":
            rng = np.random.default_rng(CFG["split_state"])
            _TEST_CACHE[name] = set(rng.choice(N, size=max(1, round(CFG["test_frac"] * N)), replace=False).tolist())
        elif name == "cs2_hubs":
            g = get_graph("cs2", 3)
            comms = louvain_communities(g, seed=42)
            _TEST_CACHE[name] = gravity_per_community(g, comms, (0.33, 0.33, 0.33))[1]
        else:
            raise ValueError(name)
        print(f"test set '{name}': {len(_TEST_CACHE[name])} MOFs")
    return _TEST_CACHE[name]


def make_splits(test_nodes, seed):
    rng = np.random.RandomState(seed)
    remaining = np.array([i for i in range(N) if i not in test_nodes])
    rng.shuffle(remaining)
    n_val = max(1, int(len(remaining) * CFG["val_frac"]))
    return remaining[n_val:], remaining[:n_val], np.array(sorted(test_nodes))

## 6. Sparsification: Black Hole, random, k-center

In [7]:
def black_hole_nodes(graph, gravity, communities, threshold, fixed_test, pld_categories):
    g = graph.copy()
    selected = set(fixed_test)
    n_total = g.number_of_nodes()
    target = int(max(0.2 * n_total, (1 - threshold) * n_total))
    target_non_test = max(0, target - len(fixed_test))
    sizes = [len(c) for c in communities]
    total = sum(sizes) or 1
    targets = [max(1, int(target_non_test * (s / total))) for s in sizes]
    diff = target_non_test - sum(targets)
    for i in range(abs(diff)):
        j = i % len(targets)
        targets[j] = max(0, targets[j] + (1 if diff > 0 else -1))
    remove = []
    for community, tcount in zip(communities, targets):
        nodes = [n for n in community if n in gravity and n not in fixed_test]
        if not nodes:
            continue
        props = pld_categories.iloc[nodes].value_counts(normalize=True).to_dict()
        for cat, prop in props.items():
            cand = [n for n in nodes if pld_categories.iloc[n] == cat]
            selected.update(sorted(cand, key=lambda n: gravity[n], reverse=True)[:max(1, int(tcount * prop))])
        remove.extend(n for n in nodes if n not in selected)
    g.remove_nodes_from(remove)
    return g


def prune_edges(graph, edge_threshold, fixed_test):
    edges = [(u, v, d["weight"]) for u, v, d in graph.edges(data=True)]
    if not edges:
        return graph
    test_edges = [e for e in edges if e[0] in fixed_test or e[1] in fixed_test]
    other = [e for e in edges if e[0] not in fixed_test and e[1] not in fixed_test]
    best = {}
    for u, v, w in test_edges:
        for n in (u, v):
            if n in fixed_test and (n not in best or w > best[n][2]):
                best[n] = (u, v, w)
    keep_test = list({(min(u, v), max(u, v)): (u, v, w) for u, v, w in best.values()}.values())
    n_keep = max(0, max(10, int((1 - edge_threshold) * len(edges))) - len(keep_test))
    other.sort(key=lambda e: e[2], reverse=True)
    out = nx.Graph()
    out.add_nodes_from(graph.nodes())
    out.add_weighted_edges_from(keep_test + other[:n_keep])
    return out


def restore_test_nodes(pruned, original, fixed_test):
    """Add back any test node pruning dropped (with its original edges) so every method is
    evaluated on the identical test set."""
    missing = fixed_test - set(pruned.nodes())
    if not missing:
        return pruned
    out = pruned.copy()
    out.add_nodes_from(missing)
    for node in missing:
        for nb, d in original.adj[node].items():
            if nb in out:
                out.add_edge(node, nb, **d)
    return out


def kcenter_nodes(features, n_keep, seed, required):
    rng = np.random.default_rng(seed)
    selected = list(dict.fromkeys(required)) or [int(rng.integers(0, N))]
    min_dist = np.full(N, np.inf)
    for c in selected:
        min_dist = np.minimum(min_dist, np.linalg.norm(features - features[c], axis=1))
    while len(selected) < n_keep and len(selected) < N:
        nxt = int(np.argmax(min_dist))
        if min_dist[nxt] == 0:
            break
        selected.append(nxt)
        min_dist = np.minimum(min_dist, np.linalg.norm(features - features[nxt], axis=1))
        min_dist[nxt] = 0.0
    short = min(n_keep, N) - len(selected)
    if short > 0:  # many MOFs share one linker -> fewer distinct points than the budget: fill randomly
        rest = np.setdiff1d(np.arange(N), np.array(selected, dtype=int))
        selected.extend(int(i) for i in rng.choice(rest, size=short, replace=False))
    return selected[:n_keep]


def kcenter_space(graph_type, lm):
    fpn = _normalize(FP)
    if graph_type in ("cs2", "fingerprint"):
        return fpn
    if graph_type == "semantic":
        return _normalize(EMB[lm])
    return np.concatenate([fpn, _normalize(EMB[lm])], axis=1)


_BH_CACHE = {}


def sparsify(graph, graph_type, k, beta, lm, method, retention, seed, test_name, bh_weights):
    test_nodes = get_test_nodes(test_name)
    if method == "none":
        return graph
    if method == "black_hole":
        key = (graph_type, k, beta, lm, retention, bh_weights, test_name)
        if key not in _BH_CACHE:  # deterministic (Louvain seed 42), so computed once, reused for every seed
            communities = louvain_communities(graph, seed=42)
            gravity, own_test = gravity_per_community(graph, communities, bh_weights)
            pruned = black_hole_nodes(graph, gravity, communities, 1 - retention, own_test, df["pld_category"])
            pruned = prune_edges(pruned, 1 - retention, own_test)
            _BH_CACHE[key] = restore_test_nodes(pruned, graph, test_nodes)
        return _BH_CACHE[key]
    n_keep = max(len(test_nodes), round(retention * N))
    if method == "random":
        rng = np.random.default_rng(seed)
        cand = [n for n in graph.nodes() if n not in test_nodes]
        extra = rng.choice(cand, size=min(max(0, n_keep - len(test_nodes)), len(cand)), replace=False).tolist()
        return graph.subgraph(set(test_nodes) | set(extra)).copy()
    if method == "k_center":
        chosen = kcenter_nodes(kcenter_space(graph_type, lm), n_keep, seed, sorted(test_nodes))
        return graph.subgraph(set(chosen)).copy()
    raise ValueError(method)

## 7. Models

`residual_gcn` is the new variant: it starts as a plain MLP and the graph term is multiplied by a learnable
gate that starts at 0, so each MOF keeps trusting its own features unless neighbours demonstrably help.

In [8]:
class GCN(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.conv1, self.conv2 = GCNConv(d_in, d_h), GCNConv(d_h, d_out)

    def forward(self, x, ei, ew=None):
        h = F.dropout(F.relu(self.conv1(x, ei, ew)), p=0.3, training=self.training)
        return self.conv2(h, ei, ew)


class GraphSAGE(nn.Module):
    """Mean aggregation over neighbours + self loop (same hand-rolled layer as the repo)."""

    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.l1, self.l2 = nn.Linear(d_in, d_h), nn.Linear(d_h, d_out)

    def _norm_adj(self, n, ei, ew, device):
        loops = torch.stack([torch.arange(n, device=device)] * 2)
        if ew is None:
            ew = torch.ones(ei.size(1), device=device)
            self_w = torch.ones(n, device=device)
        else:
            self_w = torch.ones(n, device=device) * ew.mean()
        ei2, ew2 = torch.cat([ei, loops], 1), torch.cat([ew, self_w])
        adj = torch.sparse_coo_tensor(ei2, ew2, (n, n))
        deg = torch.sparse.sum(adj, dim=1).to_dense().clamp(min=1.0)
        return torch.sparse_coo_tensor(ei2, ew2 / deg[ei2[0]], (n, n))

    def forward(self, x, ei, ew=None):
        A = self._norm_adj(x.size(0), ei, ew, x.device)
        h = F.dropout(F.relu(self.l1(torch.sparse.mm(A, x))), p=0.3, training=self.training)
        return self.l2(torch.sparse.mm(A, h))


class GAT(nn.Module):
    def __init__(self, d_in, d_h, d_out, heads=12):
        super().__init__()
        self.conv1 = GATConv(d_in, d_h, heads=heads, dropout=0.3)
        self.norm = nn.LayerNorm(d_h * heads)
        self.conv2 = GATConv(d_h * heads, d_out, heads=1, concat=False, dropout=0.3)

    def forward(self, x, ei, ew=None):
        h = F.dropout(F.relu(self.norm(self.conv1(x, ei))), p=0.3, training=self.training)
        return self.conv2(h, ei)


class ResidualGCN(nn.Module):
    def __init__(self, d_in, d_h, d_out):
        super().__init__()
        self.lin_in, self.conv = nn.Linear(d_in, d_h), GCNConv(d_h, d_h)
        self.gate = nn.Parameter(torch.zeros(1))  # 0 -> starts as a pure MLP
        self.lin_out = nn.Linear(d_h, d_out)

    def forward(self, x, ei, ew=None):
        h = F.dropout(F.relu(self.lin_in(x)), p=0.3, training=self.training)
        h = h + self.gate * F.relu(self.conv(h, ei, ew))
        return self.lin_out(h)


GNN_CLASSES = {"gcn": GCN, "graphsage": GraphSAGE, "gat": GAT, "residual_gcn": ResidualGCN}


def edge_arrays(graph):
    edges = list(graph.edges(data=True))
    if not edges:
        return np.zeros((2, 0), np.int64), np.zeros(0, np.float32)
    src = [u for u, v, d in edges] + [v for u, v, d in edges]
    dst = [v for u, v, d in edges] + [u for u, v, d in edges]
    return np.array([src, dst]), np.array([d["weight"] for u, v, d in edges] * 2, np.float32)


def train_eval_gnn(model_name, feats, graph, test_nodes, seed):
    train_idx, val_idx, test_idx = make_splits(test_nodes, seed)
    ei_np, ew_np = edge_arrays(graph)
    x = torch.tensor(feats, dtype=torch.float, device=DEVICE)
    y = torch.tensor(LABELS, dtype=torch.float, device=DEVICE)
    ei = torch.tensor(ei_np, dtype=torch.long, device=DEVICE)
    ew = torch.tensor(ew_np, dtype=torch.float, device=DEVICE) if ew_np.size else None
    tr, va, te = (torch.tensor(a, dtype=torch.long, device=DEVICE) for a in (train_idx, val_idx, test_idx))

    model = GNN_CLASSES[model_name](feats.shape[1], CFG["hidden"], 1).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    best_val, best_state, wait, epochs_run = float("inf"), None, 0, 0
    t0 = time.time()
    for epoch in range(CFG["epochs"]):
        model.train()
        opt.zero_grad()
        loss = F.mse_loss(model(x, ei, ew).squeeze(-1)[tr], y[tr])
        loss.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            val_loss = F.mse_loss(model(x, ei, ew).squeeze(-1)[va], y[va]).item()
        epochs_run = epoch + 1
        if val_loss < best_val:
            best_val, wait = val_loss, 0
            best_state = {k_: v.clone() for k_, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= CFG["patience"]:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(x, ei, ew).squeeze(-1)[te].cpu().numpy()
    true = LABELS[test_idx]
    out = metrics(true, pred)
    out.update(epochs_run=epochs_run, best_val_mse=best_val, train_seconds=time.time() - t0)
    if model_name == "residual_gcn":
        out["learned_gate"] = float(model.gate.item())
    return out


def metrics(true, pred):
    return {"mae": float(mean_absolute_error(true, pred)),
            "rmse": float(np.sqrt(mean_squared_error(true, pred))),
            "r2": float(r2_score(true, pred))}


def train_eval_classic(model_name, feats, test_nodes, seed):
    train_idx, _val_idx, test_idx = make_splits(test_nodes, seed)
    t0 = time.time()
    if model_name == "random_forest":
        model = RandomForestRegressor(n_estimators=CFG["rf_trees"], random_state=seed, n_jobs=-1)
    elif model_name == "mlp":
        model = MLPRegressor(hidden_layer_sizes=CFG["mlp_hidden"], random_state=seed,
                             early_stopping=True, n_iter_no_change=20, max_iter=500)
    elif model_name == "knn":
        model = KNeighborsRegressor(n_neighbors=CFG["knn_neighbors"], n_jobs=-1)
    else:
        raise ValueError(model_name)
    model.fit(feats[train_idx], LABELS[train_idx])
    out = metrics(LABELS[test_idx], model.predict(feats[test_idx]))
    out["train_seconds"] = time.time() - t0
    return out

## 8. Build the list of runs

In [9]:
def feature_matrix(name, lm, scaling, test_nodes):
    if name == "fingerprint":
        feats = X_FP
    elif name == "embedding":
        feats = EMB[lm]
    elif name == "fingerprint+embedding":
        feats = np.hstack([X_FP, EMB[lm]])
    else:
        raise ValueError(name)
    if scaling == "standard":  # statistics from non-test MOFs only; no labels involved
        non_test = np.array([i for i in range(N) if i not in test_nodes])
        feats = StandardScaler().fit(feats[non_test]).transform(feats).astype(np.float32)
    return np.ascontiguousarray(feats, dtype=np.float32)


def uses_lm(graph_type, node_features):
    return graph_type in ("semantic", "hybrid") or "embedding" in node_features


def make_specs(cfg):
    specs, seen = [], set()

    def add(**kw):
        base = dict(block=None, model=None, node_features="fingerprint", scaling="none", graph_type="none", k=0,
                    beta=0.0, language_model="-", sparsify="none", retention=1.0, bh_weights="-", test_set=None)
        base.update(kw)
        if base["graph_type"] != "hybrid":
            base["beta"] = 0.0
        for seed in cfg["seeds"]:
            spec = dict(base, seed=seed)
            rid = hashlib.md5(json.dumps(spec, sort_keys=True).encode()).hexdigest()[:16]
            if rid not in seen:
                seen.add(rid)
                specs.append(dict(spec, run_id=rid))

    def lms(graph_type, node_features):
        return cfg["language_models"] if uses_lm(graph_type, node_features) else ["-"]

    def betas(graph_type):
        return cfg["hybrid_betas"] if graph_type == "hybrid" else [0.0]

    for ts in cfg["test_sets"]:
        for sc in cfg["scalings"]:
            if cfg["block_non_graph"]:
                for m in cfg["non_graph_models"]:
                    for fs in cfg["feature_sets"]:
                        for lm in lms("none", fs):
                            add(block="non_graph", model=m, node_features=fs, scaling=sc, language_model=lm, test_set=ts)
            if cfg["block_no_edge"]:
                for m in cfg["gnn_models"]:
                    for fs in cfg["feature_sets"]:
                        for lm in lms("none", fs):
                            add(block="no_edge", model=m, node_features=fs, scaling=sc, language_model=lm, test_set=ts)
            if cfg["block_graph_full"]:
                for gt in cfg["graph_types"]:
                    for k in cfg["k_values"]:
                        for beta in betas(gt):
                            for nf in cfg["node_features"]:
                                for lm in lms(gt, nf):
                                    for m in cfg["gnn_models"]:
                                        add(block="graph_full", model=m, node_features=nf, scaling=sc, graph_type=gt, k=k,
                                            beta=beta, language_model=lm, test_set=ts)
            if cfg["block_sparsify"]:
                for gt in cfg["sparsify_graphs"]:
                    for k in cfg["sparsify_k_values"]:
                        for beta in betas(gt):
                            for nf in cfg["node_features"]:
                                for lm in lms(gt, nf):
                                    for method in cfg["sparsify_methods"]:
                                        for ret in cfg["retentions"]:
                                            weightings = cfg["bh_weights"] if method == "black_hole" else [None]
                                            for w in weightings:
                                                wlabel = "/".join(str(v) for v in w) if w else "-"
                                                for m in cfg["gnn_models"]:
                                                    add(block="sparsify", model=m, node_features=nf, scaling=sc, graph_type=gt,
                                                        k=k, beta=beta, language_model=lm, sparsify=method, retention=ret,
                                                        bh_weights=wlabel, test_set=ts)
    return specs


SPECS = make_specs(CFG)
spec_df = pd.DataFrame(SPECS)
print(f"TOTAL RUNS: {len(SPECS)}")
print(spec_df.groupby(["block", "model"]).size().unstack(fill_value=0))

TOTAL RUNS: 990
model       gcn  graphsage  knn  mlp  random_forest  residual_gcn
block                                                            
graph_full   60         60    0    0              0            60
no_edge      15         15    0    0              0            15
non_graph     0          0   15   15             15             0
sparsify    240        240    0    0              0           240


## 9. Run (resumable)

Each finished run is appended to `per_run_results.csv` immediately. Re-running this cell skips runs that are already there.

In [10]:
os.makedirs(OUT_DIR, exist_ok=True)
PER_RUN_PATH = os.path.join(OUT_DIR, "per_run_results.csv")
done_ids = set(pd.read_csv(PER_RUN_PATH)["run_id"]) if os.path.exists(PER_RUN_PATH) else set()
todo = [s for s in SPECS if s["run_id"] not in done_ids]
print(f"already done: {len(done_ids & {s['run_id'] for s in SPECS})}  |  to do now: {len(todo)}")


# fixed column order, so rows from different model types (RF has no epochs, residual GCN has a gate, ...) line up
PER_RUN_COLUMNS = ["run_id", "block", "model", "node_features", "scaling", "graph_type", "k", "beta", "language_model",
                   "sparsify", "retention", "bh_weights", "test_set", "seed", "mae", "rmse", "r2", "train_seconds",
                   "epochs_run", "best_val_mse", "learned_gate", "n_test", "n_features", "graph_nodes",
                   "graph_nodes_with_edges", "graph_edges", "device", "data_sha256", "finished_at"]


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def execute(spec):
    set_all_seeds(spec["seed"])
    test_nodes = get_test_nodes(spec["test_set"])
    lm = spec["language_model"]
    feats = feature_matrix(spec["node_features"], lm, spec["scaling"], test_nodes)
    if spec["block"] == "non_graph":
        res = train_eval_classic(spec["model"], feats, test_nodes, spec["seed"])
        graph = None
    else:
        base = get_graph(spec["graph_type"], spec["k"], spec["beta"], lm)
        wt = tuple(float(v) for v in spec["bh_weights"].split("/")) if spec["bh_weights"] != "-" else None
        graph = sparsify(base, spec["graph_type"], spec["k"], spec["beta"], lm, spec["sparsify"], spec["retention"],
                         spec["seed"], spec["test_set"], wt)
        res = train_eval_gnn(spec["model"], feats, graph, test_nodes, spec["seed"])
    row = dict(spec)
    row.update(res)
    row.update(n_test=len(test_nodes), n_features=feats.shape[1], device=str(DEVICE),
               graph_nodes_with_edges=int(sum(1 for _, d in graph.degree() if d > 0)) if graph is not None else 0,
               graph_nodes=graph.number_of_nodes() if graph is not None else N,
               graph_edges=graph.number_of_edges() if graph is not None else 0,
               data_sha256=DATA_SHA[:16], finished_at=datetime.now().isoformat(timespec="seconds"))
    return row


START = time.time()
pbar = tqdm(todo, desc="runs")
for spec in pbar:
    if CFG["max_hours"] and (time.time() - START) / 3600 > CFG["max_hours"]:
        print(f"max_hours={CFG['max_hours']} reached — stopping. Re-run this cell to resume.")
        break
    try:
        row = execute(spec)
    except Exception as exc:  # keep going; the failure is recorded and can be retried by re-running
        print(f"FAILED {spec['model']} {spec['graph_type']} k={spec['k']} {spec['sparsify']}: {type(exc).__name__}: {exc}")
        with open(os.path.join(OUT_DIR, "failed_runs.log"), "a") as fh:
            fh.write(f"{datetime.now().isoformat()} {json.dumps(spec)} {type(exc).__name__}: {exc}\n")
        continue
    pd.DataFrame([row]).reindex(columns=PER_RUN_COLUMNS).to_csv(
        PER_RUN_PATH, mode="a", header=not os.path.exists(PER_RUN_PATH), index=False)
    pbar.set_postfix(model=row["model"], graph=row["graph_type"], r2=round(row["r2"], 3))
print(f"finished in {(time.time() - START) / 60:.1f} min")

already done: 0  |  to do now: 990


runs:   0%|          | 0/990 [00:00<?, ?it/s]

test set 'random15': 2144 MOFs
built graph ('cs2', 3, 0.0, '-'): 32988 edges in 3s
built graph ('cs2', 5, 0.0, '-'): 54238 edges in 3s
built graph ('cs2', 10, 0.0, '-'): 107706 edges in 3s
built graph ('fingerprint', 3, 0.0, '-'): 36576 edges in 3s
built graph ('fingerprint', 5, 0.0, '-'): 59400 edges in 3s
built graph ('fingerprint', 10, 0.0, '-'): 115294 edges in 3s
built graph ('semantic', 3, 0.0, 'chemberta'): 36248 edges in 3s
built graph ('semantic', 5, 0.0, 'chemberta'): 58905 edges in 2s
built graph ('semantic', 10, 0.0, 'chemberta'): 113658 edges in 2s
built graph ('hybrid', 3, 0.5, 'chemberta'): 36258 edges in 3s
built graph ('hybrid', 5, 0.5, 'chemberta'): 58712 edges in 3s
built graph ('hybrid', 10, 0.5, 'chemberta'): 113776 edges in 3s
finished in 360.8 min


## 10. Aggregate, compare with the no-edge reference, export

In [11]:
per_run = pd.read_csv(PER_RUN_PATH)
GROUP = ["block", "model", "node_features", "scaling", "graph_type", "k", "beta", "language_model",
         "sparsify", "retention", "bh_weights", "test_set"]
agg = (per_run.groupby(GROUP, dropna=False)
       .agg(n_seeds=("seed", "nunique"), mae_mean=("mae", "mean"), mae_std=("mae", "std"),
            rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), r2_mean=("r2", "mean"), r2_std=("r2", "std"),
            seconds_mean=("train_seconds", "mean"), graph_nodes=("graph_nodes", "mean"),
            graph_edges=("graph_edges", "mean"))
       .reset_index())

# "Does the graph help or hurt?": R2 of each graph run minus R2 of the SAME model/features/scaling/test set with no edges.
# The graph's own language model is ignored in the match unless the node features contain embeddings.
agg["ref_lm"] = np.where(agg["node_features"].str.contains("embedding"), agg["language_model"], "-")
ref_keys = ["model", "node_features", "scaling", "ref_lm", "test_set"]
no_edge = agg[agg["block"] == "no_edge"][ref_keys + ["r2_mean", "mae_mean"]].rename(
    columns={"r2_mean": "r2_no_edge", "mae_mean": "mae_no_edge"})
vs = agg[agg["block"].isin(["graph_full", "sparsify"])].merge(no_edge, on=ref_keys, how="left")
vs["r2_change_vs_no_edge"] = vs["r2_mean"] - vs["r2_no_edge"]
vs["verdict"] = np.where(vs["r2_change_vs_no_edge"] > 0.01, "graph helps",
                         np.where(vs["r2_change_vs_no_edge"] < -0.01, "graph hurts", "about the same"))

best_nongraph = agg[agg["block"] == "non_graph"].sort_values("r2_mean", ascending=False).head(1)
print("Best non-graph model:")
print(best_nongraph[["model", "node_features", "r2_mean", "mae_mean"]].to_string(index=False))
print("\nTop 15 GNN configurations by R2:")
print(agg[agg["block"] != "non_graph"].sort_values("r2_mean", ascending=False).head(15)[
    ["block", "model", "graph_type", "k", "sparsify", "retention", "node_features", "r2_mean", "r2_std", "mae_mean"]
].to_string(index=False))
print("\nGraph vs no-edge verdict counts:")
print(vs["verdict"].value_counts().to_string())

agg.to_csv(os.path.join(OUT_DIR, "aggregated_results.csv"), index=False)
vs.to_csv(os.path.join(OUT_DIR, "graph_vs_no_graph.csv"), index=False)

Best non-graph model:
        model node_features  r2_mean  mae_mean
random_forest   fingerprint 0.882536  0.521627

Top 15 GNN configurations by R2:
     block        model  graph_type  k   sparsify  retention         node_features  r2_mean   r2_std  mae_mean
graph_full residual_gcn         cs2  3       none        1.0           fingerprint 0.844781 0.010184  0.729532
graph_full residual_gcn         cs2 10       none        1.0           fingerprint 0.844061 0.007626  0.732545
graph_full residual_gcn      hybrid  3       none        1.0           fingerprint 0.843506 0.008177  0.731711
graph_full residual_gcn         cs2  5       none        1.0           fingerprint 0.842515 0.007304  0.735072
graph_full residual_gcn    semantic  3       none        1.0           fingerprint 0.841485 0.006341  0.737381
graph_full residual_gcn    semantic  5       none        1.0           fingerprint 0.841167 0.006670  0.739952
graph_full residual_gcn    semantic 10       none        1.0           fi

## 11. Metadata, README, Excel and zip  — download `mofsemnet_results.zip`

In [12]:
def _pkg(name):
    try:
        return __import__(name).__version__
    except Exception:
        return "?"


meta = dict(
    created=datetime.now().isoformat(timespec="seconds"), profile=PROFILE,
    config={k_: (list(v) if isinstance(v, tuple) else v) for k_, v in CFG.items()},
    data_file=DATA_PATH, data_sha256=DATA_SHA, n_mofs=N, seeds=CFG["seeds"], device=str(DEVICE),
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    python=sys.version.split()[0], platform=platform.platform(),
    packages={p: _pkg(p) for p in ["torch", "torch_geometric", "networkx", "sklearn", "pandas", "numpy", "rdkit"]},
    embedding_files={lm: (EMB_PATHS[lm], file_sha256(EMB_PATHS[lm])[:16]) for lm in CFG["language_models"]
                     if os.path.exists(EMB_PATHS[lm])},
    runs_total=len(SPECS), runs_in_csv=len(per_run),
)
with open(os.path.join(OUT_DIR, "run_metadata.json"), "w") as fh:
    json.dump(meta, fh, indent=2, default=str)

readme = f"""MOFSemNet server run — how these files were made and how to regenerate them
=============================================================================
Notebook      : the single notebook this folder came from (profile = {PROFILE})
Data          : {DATA_PATH}  (SHA-256 {DATA_SHA})
Seeds         : {CFG['seeds']}  (every run seeds python, numpy and torch; splits use the same seed)
Test sets     : {CFG['test_sets']}  (random15 = 15% random MOFs, fixed with split_state={CFG['split_state']};
                cs2_hubs = Case Study 2's high-gravity Louvain-community test set)
Target        : continuous Pore Limiting Diameter (Angstrom); metrics MAE, RMSE, R2 on the held-out test set.

Files
  per_run_results.csv     one row per (configuration, seed): all settings, metrics, timing, graph size
  aggregated_results.csv  mean and std over seeds per configuration
  graph_vs_no_graph.csv   each graph result compared with the same model run WITHOUT edges
  run_metadata.json       full configuration, package versions, GPU, data hash
  mofsemnet_results.xlsx  all of the above as sheets

To regenerate: put MOFCSD.csv (and the embeddings .npz files) next to the notebook, set PROFILE = "{PROFILE}"
and the CFG overrides recorded in run_metadata.json, then Run All. Delete per_run_results.csv first
for a clean run (otherwise finished runs are skipped = resume).
"""
with open(os.path.join(OUT_DIR, "README.txt"), "w") as fh:
    fh.write(readme)

xlsx_path = os.path.join(OUT_DIR, "mofsemnet_results.xlsx")
try:
    with pd.ExcelWriter(xlsx_path) as xw:
        agg.to_excel(xw, sheet_name="aggregated", index=False)
        vs.to_excel(xw, sheet_name="graph_vs_no_graph", index=False)
        per_run.to_excel(xw, sheet_name="per_run", index=False)
        pd.DataFrame([{"key": k_, "value": json.dumps(v, default=str)} for k_, v in meta.items()]).to_excel(
            xw, sheet_name="metadata", index=False)
    print("wrote", xlsx_path)
except Exception as exc:
    print(f"Excel export skipped ({type(exc).__name__}: {exc}). Run `pip install openpyxl` and re-run this cell; the CSVs are complete.")

zip_path = "mofsemnet_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir(OUT_DIR)):
        zf.write(os.path.join(OUT_DIR, name), arcname=name)
print("download:", os.path.abspath(zip_path))

wrote mofsemnet_results/mofsemnet_results.xlsx
download: /home/jovyan/mofsemnet_results.zip
